In [1]:
from __future__ import annotations

import json
import logging
import os
import re
from io import BytesIO
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from google import genai
from google.api_core.exceptions import NotFound
from google.cloud import bigquery, storage
from google.genai import types
from google.oauth2 import service_account
from pypdf import PdfReader, PdfWriter


def generate_pillar_summary(
    drug_name: str,
    gcs_path: str | None = None,
    project_id: str | None = None,
    credentials_path: str | None = None,
    bucket_name: str = "cognito-gcs",
    model_name: str = "gemini-3-flash-preview",
    log_level: int = logging.INFO,
) -> dict[str, Any]:
    """
    Generate a molecule-level patent/market narrative summary from report PDFs and metadata.

    This function also counts estimated tokens consumed for the prompt+PDF inputs and model text output
    and returns `input_tokens`, `output_tokens`, and `total_tokens` in the result dict.
    """
    load_dotenv()

    logger = logging.getLogger("generate_pillar_summary")
    if not logger.handlers:
        logging.basicConfig(
            level=log_level,
            format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
        )
    logger.setLevel(log_level)

    if not drug_name or not drug_name.strip():
        raise ValueError("drug_name must not be empty")

    project_id = project_id or os.getenv("PROJECT_ID")
    credentials_path = credentials_path or os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

    if not project_id:
        raise ValueError("PROJECT_ID is required. Set it in .env or pass project_id explicitly.")

    prompt_template = """
{input_json}

You are a pharmaceutical patent and competitive intelligence analyst.

Your task is to generate a SINGLE narrative summary for a molecule using information extracted from the attached PDF documents and the provided JSON input.

OBJECTIVE
Create a concise, executive-ready product narrative that summarizes the molecule's patent and market position.

OUTPUT REQUIREMENTS
- Output exactly ONE paragraph.
- Maximum 100 words (hard cap).
- Write as a natural product narrative, not as bullets or a structured report.
- Do NOT mention Cognito dimensions, pillar names, scores, frameworks, or source categories.
- Use factual information only from the supplied documents and JSON.
- Do not invent or assume facts.
- If multiple sources provide similar information, consolidate them.
- Use plain ASCII punctuation only (for example: - instead of em dash, straight quotes, and ... instead of ellipsis).


MANDATORY CONTENT TO INCLUDE
1. Molecule name.
2. Mention the Dosage form(s) & Dosage strength(s) and delivery mechanism (only if available in reports)
4. Indications it addresses.
5. From the input data, if it is a marketed drug mention it as marketed, or if it is in pre-registration mention it. Otherwise mention it as clinical with current phase.
6. Comment on entry difficulty or exclusivity outlook by referring to the `Primary Market Entry Horizon.pdf`. 
    - If from the `Primary Market Entry Horizon.pdf` you see that exclusivity is expired in any of the primary market, mention it along with in which market it's exclusivity has expired
7.Comment on patent thicket based on the executive summary from `Patent Strength and Invalidity Opportunity.pdf`
    - Do no mention or refer to the number of patents figure from the reports (You will be penalized for that) - Just comment on wheter it is dense, moderate or sparse thicket similar to what is given in example output
8. Comment on best opportunity for competitors (for example, geographic gaps, design-around potentials, circumvention potential) from `Global Launch Sequencing & Arbitrage.pdf` & `Patent Thicket and Circumvention Feasibility.pdf`
9. Overall outlook verdict as one simple sentence (Whether it is Good/ Moderate/ Challenging).

Note: Keep the summary generic and do not cite anything specific like exclusivity year or name of the geography or any particular strategies or pathways from the attached reports. You will be strictly penalized for this

SPECIAL GUIDANCE

For CLINICAL assets:
- Describe the patent estate and entry horizon.
- Highlight design-around and geographic opportunities.
- State the primary patent barrier.

For COMMERCIAL assets:
- Describe marketed status and indications.
- Highlight the best challenger pathway.
- State the primary patent barrier.

INPUTS
1. Attached PDF documents = supporting evidence.
2. JSON = structured metadata and assessment outputs.

OUTPUT
Return only the final single paragraph and absolutely nothing else. 

Example Output: 
For Clinical - `Bofanglutide is a Phase 3 GLP-1 receptor agonist — bi-weekly subcutaneous injection (illustrative strengths 2 mg / 4 mg) — protected by a layered patent estate spanning core peptide, formulation, and lifecycle-management claims across major markets. Broad coverage creates a challenging market-entry horizon, with value most likely from geographic protection gaps and targeted design-around opportunities rather than direct near-term challenges. The core peptide patent is the primary barrier. Overall IP outlook is Challenging.`
For Commercial - `Tirzepatide is a marketed dual GIP/GLP-1 receptor agonist — once-weekly subcutaneous injection (2.5-15 mg) for Type 2 diabetes and obesity — protected by a dense estate of composition, formulation, dosing-regimen, and device patents extending well into the next decade. Broad multi-layer coverage and long US/EU exclusivity make near-term entry highly constrained, with value limited to selective geographic gaps and long-dated design-around pathways. The core composition and dosing patents are the primary barrier. Overall IP outlook is Moderate.`
"""

    def get_gcs_client() -> storage.Client:
        if credentials_path and Path(credentials_path).exists():
            logger.info("Using explicit service-account credentials for GCS")
            return storage.Client.from_service_account_json(credentials_path)
        logger.info("Using ADC/project credentials for GCS")
        return storage.Client(project=project_id)

    def get_bq_client() -> bigquery.Client:
        if credentials_path and Path(credentials_path).exists():
            logger.info("Using explicit service-account credentials for BigQuery")
            credentials = service_account.Credentials.from_service_account_file(credentials_path)
            return bigquery.Client(project=project_id, credentials=credentials)
        logger.info("Using ADC/project credentials for BigQuery")
        return bigquery.Client(project=project_id)

    def normalize_gcs_root(path: str) -> str:
        cleaned = (path or "").strip().rstrip("/")
        if cleaned.startswith("gs://"):
            cleaned = cleaned[len("gs://"):]
            if cleaned.startswith(f"{bucket_name}/"):
                cleaned = cleaned[len(bucket_name) + 1 :]
            elif "/" in cleaned:
                cleaned = cleaned.split("/", 1)[1]
        elif cleaned.startswith(f"{bucket_name}/"):
            cleaned = cleaned[len(bucket_name) + 1 :]
        return cleaned.strip("/")

    def classify_asset(stages: list[str]) -> str:
        stage_text = " ".join(stages).lower()
        if "pre-registration" in stage_text or "marketed" in stage_text:
            return "commercial"
        return "clinical"

    def extract_stage_metadata(stages: list[str]) -> dict[str, Any]:
        normalized = [str(stage).strip() for stage in stages if str(stage).strip()]
        lower_values = [stage.lower() for stage in normalized]

        is_marketed = any("marketed" in stage for stage in lower_values)
        is_pre_registered = any("pre-registration" in stage or "pre registration" in stage for stage in lower_values)

        if is_marketed:
            status_label = "marketed"
        elif is_pre_registered:
            status_label = "pre-registered"
        else:
            status_label = "clinical"

        clinical_stages = [
            stage for stage in normalized
            if "marketed" not in stage.lower()
            and "pre-registration" not in stage.lower()
            and "pre registration" not in stage.lower()
        ]

        current_phase = clinical_stages[0] if clinical_stages else None

        return {
            "stage_status": status_label,
            "is_marketed": is_marketed,
            "is_pre_registered": is_pre_registered,
            "current_phase": current_phase,
            "highest_development_stages": normalized,
        }

    def build_prompt(input_payload: dict[str, Any]) -> str:
        input_json = json.dumps(input_payload, indent=2, default=str)
        return prompt_template.replace("{input_json}", input_json)

    def normalize_output_text(text: str) -> str:
        repaired = (text or "").strip()

        # Fix common UTF-8/Windows-1252 mojibake sequences.
        mojibake_map = {
            "â€”": "—",
            "â€“": "–",
            "â€˜": "‘",
            "â€™": "’",
            "â€œ": "“",
            "â€": "”",
            "â€¦": "…",
            "Â": "",
        }
        for bad, good in mojibake_map.items():
            repaired = repaired.replace(bad, good)

        # Normalize punctuation to ASCII for downstream systems that mis-handle Unicode punctuation.
        ascii_punctuation_map = {
            "—": " - ",
            "–": " - ",
            "‘": "'",
            "’": "'",
            "“": "\"",
            "”": "\"",
            "…": "...",
        }
        for src, dest in ascii_punctuation_map.items():
            repaired = repaired.replace(src, dest)

        return re.sub(r"\s+", " ", repaired).strip()

    def parse_gemini_output(raw_text: str) -> dict[str, Any]:
        cleaned_text = normalize_output_text(raw_text)
        try:
            parsed = json.loads(cleaned_text)
            if isinstance(parsed, dict):
                if isinstance(parsed.get("summary"), str):
                    parsed["summary"] = normalize_output_text(parsed["summary"])
                if isinstance(parsed.get("text"), str):
                    parsed["text"] = normalize_output_text(parsed["text"])
                return parsed
            if isinstance(parsed, str):
                return {"summary": normalize_output_text(parsed)}
            return {"summary": parsed}
        except json.JSONDecodeError:
            return {"summary": cleaned_text}

    def get_pdf_names() -> list[str]:
        common_reports = [
            "Primary Market Entry Horizon.pdf",
            "Global Launch Sequencing & Arbitrage.pdf",
            "Patent Strength and Invalidity Opportunity.pdf",
            "Patent Thicket and Circumvention Feasibility.pdf",
        ]
        return common_reports

    def extract_first_page_pdf(pdf_bytes: bytes) -> bytes:
        reader = PdfReader(BytesIO(pdf_bytes))
        if len(reader.pages) == 0:
            raise ValueError("PDF has no pages")
        writer = PdfWriter()
        writer.add_page(reader.pages[0])
        output_buffer = BytesIO()
        writer.write(output_buffer)
        return output_buffer.getvalue()

    def get_report_first_page_payloads(base_path: str, report_names: list[str]) -> tuple[list[tuple[str, bytes]], list[str]]:
        gcs_client = get_gcs_client()
        bucket = gcs_client.bucket(bucket_name)
        root = normalize_gcs_root(base_path)

        if not root:
            raise ValueError("gcs_path must not be empty")

        pdf_first_page_payloads: list[tuple[str, bytes]] = []
        missing_reports: list[str] = []

        for report_name in report_names:
            blob_name = f"{root}/{report_name}"
            blob = bucket.blob(blob_name)
            try:
                full_pdf_bytes = blob.download_as_bytes()
            except NotFound:
                missing_reports.append(blob_name)
                logger.warning("Report missing, skipping: %s", blob_name)
                continue

            try:
                first_page_pdf_bytes = extract_first_page_pdf(full_pdf_bytes)
                pdf_first_page_payloads.append((report_name, first_page_pdf_bytes))
                logger.info("Attached first page only for report: gs://%s/%s", bucket_name, blob_name)
            except Exception as exc:
                missing_reports.append(blob_name)
                logger.warning("Could not extract first page from %s: %s", blob_name, exc)

        return pdf_first_page_payloads, missing_reports

    sql_query = f"""
SELECT DISTINCT Highest_Development_Stage
FROM `cognito-prod-394707.cognito_prod_datamart.vw_drug_details_full`
WHERE Cleaned_Generic_Name LIKE ('%{drug_name}%')
"""

    logger.info("Querying development stages for drug: %s", drug_name)
    bq_client = get_bq_client()
    query_df = bq_client.query(sql_query).to_dataframe()
    highest_stages = [
        str(value).strip()
        for value in query_df["Highest_Development_Stage"].dropna().tolist()
    ]

    stage_metadata = extract_stage_metadata(highest_stages)

    asset_type = classify_asset(highest_stages)
    logger.info("Asset classified as: %s", asset_type)

    resolved_gcs_path = gcs_path or f"{bucket_name}/Cognito_new/reports/{drug_name}/IP/"
    pdf_names = get_pdf_names()

    input_payload = {
        "drug_name": drug_name,
        "asset_type": asset_type,
        "highest_development_stages": highest_stages,
        "stage_status": stage_metadata["stage_status"],
        "is_marketed": stage_metadata["is_marketed"],
        "is_pre_registered": stage_metadata["is_pre_registered"],
        "current_phase": stage_metadata["current_phase"],
        "pdf_names": pdf_names,
    }

    pdf_first_page_payloads, missing_reports = get_report_first_page_payloads(resolved_gcs_path, pdf_names)
    if missing_reports:
        logger.warning("Missing/invalid reports count: %d", len(missing_reports))

    if not pdf_first_page_payloads:
        logger.warning("No valid report bytes found. Proceeding with metadata-only prompt.")

    api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
    if not api_key:
        raise ValueError("GEMINI_API_KEY or GOOGLE_API_KEY is required.")

    genai_client = genai.Client(api_key=api_key)
    # genai_client = genai.Client()

    # Build prompt and parts
    prompt_text = build_prompt(input_payload)
    parts = [types.Part.from_text(text=prompt_text)]
    parts.extend(
        types.Part.from_bytes(data=pdf_bytes, mime_type="application/pdf")
        for _, pdf_bytes in pdf_first_page_payloads
    )

    # Gemini response metadata provides the authoritative token counts.
    # Call the model
    logger.info("Generating summary with model: %s", model_name)
    response = genai_client.models.generate_content(
        model=model_name,
        contents=[types.Content(role="user", parts=parts)],
        config=types.GenerateContentConfig(temperature=0),
    )

    raw_output = (response.text or "").strip()

    usage_metadata = getattr(response, "usage_metadata", None) or getattr(response, "metadata", None)
    input_tokens = int(getattr(usage_metadata, "prompt_token_count", 0) or getattr(usage_metadata, "input_token_count", 0) or 0)
    output_tokens = int(getattr(usage_metadata, "candidates_token_count", 0) or getattr(usage_metadata, "output_token_count", 0) or 0)
    total_tokens = int(getattr(usage_metadata, "total_token_count", 0) or (input_tokens + output_tokens))

    logger.info("Response token usage for %s: input=%d output=%d total=%d", drug_name, input_tokens, output_tokens, total_tokens)

    parsed_output = parse_gemini_output(raw_output)

    summary_result = {
        "drug_name": drug_name,
        "asset_type": asset_type,
        "highest_development_stages": highest_stages,
        "gcs_path": resolved_gcs_path,
        "pdf_names": pdf_names,
        "missing_reports": missing_reports,
        "raw_output": raw_output,
        "parsed_output": parsed_output,
        "input_tokens": int(input_tokens),
        "output_tokens": int(output_tokens),
        "total_tokens": int(total_tokens),
    }

    logger.info("Summary generation complete for %s (input=%d output=%d total=%d)", drug_name, input_tokens, output_tokens, total_tokens)
    return summary_result

In [2]:
# GLP-1 molecule list
full_molecules_list = ['Exenatide','Semaglutide','Albenatide','Hs-20004','Liraglutide','Pb-119','Sl-209','Semaglutide Biosimilar','Uni-E4','Cotadutide','Da-302168S','Dapiglutide','Gsbr-1290','Ecnoglutide','Insulin Degludec+Liraglutide','Lotiglipron','Vurolenatide','Pb-718','Sal-0112','Tern-601','Vct-220','Zovaglutide','Dyx-116','Efpeglenatide','Hdm-1002','Hs-10501','Mrank-001','Orforglipron Calcium','Hrs-7535','Bofanglutide','Nn-9490','Pegapamodutide','Ribupatide','Thdbh-120','Acmopatide','Hdm-1005','Mwn-109','Mazdutide','Olatorepatide','Ray-1225','Lixisenatide','Insulin Icodec+Semaglutide','Tirzepatide','Ubt-251','Na-931','Utreglutide','Efocipegtrutide','Buntanetap Tartrate+Dulaglutide+Sildenafil Citrate','Asc-30','Elecoglipron','Bgm-0504','Azd-6234+Azd-9550','Bi-3034701','Rose-010','Buntanetap Tartrate+Dulaglutide','Pemvidutide','Pegsebrenatide','Dd-01','Vk-2735','Efinopegdutide','Zenagamtide','Azd-9550','Met-224O','Te-8105','Mirikizumab+Tirzepatide','Survodutide','Rgt-075','Aleniglipron L-Arginine','Ttp-273','Ixekizumab+Tirzepatide','Brenipatide','Cagrilintide+Semaglutide','Retatrutide','Enicepatide','Naperiglipron Erbumine','Hm-15275','Berobenatide','Ct-996','Liraglutide Biobetter','Albiglutide','Benaglutide','Dulaglutide','Pegloxenatide']
molecules_list = full_molecules_list
# molecules_list = ['Semaglutide', 'Tirzepatide', 'Orforglipron Calcium', 'Bofanglutide']

In [3]:
import pandas as pd
import logging
import json
from concurrent.futures import ThreadPoolExecutor, as_completed

logger = logging.getLogger("generate_pillar_summary_runner")
logger.setLevel(logging.INFO)

results = []
# Totals across all molecules
total_input_tokens = 0
total_output_tokens = 0
total_tokens = 0

MAX_WORKERS = 7

def process_molecule(molecule):
    try:
        res = generate_pillar_summary(drug_name=molecule)
        parsed = res.get("parsed_output") if isinstance(res, dict) else None
        if isinstance(parsed, dict):
            summary_text = parsed.get("summary") or parsed.get("text") or None
        else:
            summary_text = res.get("raw_output") if isinstance(res, dict) else None
        return {
            "row": {"molecule_name": molecule, "Pillar": "IP", "Summary": summary_text},
            "input_tokens": int(res.get("input_tokens", 0)),
            "output_tokens": int(res.get("output_tokens", 0)),
            "total_tokens": int(res.get("total_tokens", 0)),
            "error": None,
        }
    except Exception as exc:
        logger.exception("Error generating summary for %s", molecule)
        return {
            "row": {"molecule_name": molecule, "Pillar": "IP", "Summary": None},
            "input_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
            "error": str(exc),
        }

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_to_molecule = {executor.submit(process_molecule, mol): mol for mol in molecules_list}
    for future in as_completed(future_to_molecule):
        outcome = future.result()
        results.append(outcome["row"])
        total_input_tokens += outcome["input_tokens"]
        total_output_tokens += outcome["output_tokens"]
        total_tokens += outcome["total_tokens"]

# Sort results to preserve original molecule order
molecule_order = {mol: i for i, mol in enumerate(molecules_list)}
results.sort(key=lambda r: molecule_order.get(r["molecule_name"], len(molecules_list)))

# Log aggregate token usage
logger.info("Aggregate token usage across %d molecules: input=%d output=%d total=%d", len(molecules_list), total_input_tokens, total_output_tokens, total_tokens)

# Optionally save token summary to disk
token_summary = {
    "molecule_count": len(molecules_list),
    "input_tokens": total_input_tokens,
    "output_tokens": total_output_tokens,
    "total_tokens": total_tokens,
}
with open('token_counts_all_molecules.json', 'w', encoding='utf-8') as f:
    json.dump(token_summary, f, indent=2)

# Final dataframe
df = pd.DataFrame(results)
df


2026-07-31 10:35:34,071 | INFO | generate_pillar_summary | Querying development stages for drug: Exenatide
2026-07-31 10:35:34,077 | INFO | generate_pillar_summary | Querying development stages for drug: Semaglutide
2026-07-31 10:35:34,089 | INFO | generate_pillar_summary | Using explicit service-account credentials for BigQuery
2026-07-31 10:35:34,089 | INFO | generate_pillar_summary | Using explicit service-account credentials for BigQuery
2026-07-31 10:35:34,092 | INFO | generate_pillar_summary | Querying development stages for drug: Albenatide
2026-07-31 10:35:34,094 | INFO | generate_pillar_summary | Querying development stages for drug: Hs-20004
2026-07-31 10:35:34,214 | INFO | generate_pillar_summary | Querying development stages for drug: Liraglutide
2026-07-31 10:35:34,224 | INFO | generate_pillar_summary | Querying development stages for drug: Sl-209
2026-07-31 10:35:34,229 | INFO | generate_pillar_summary | Querying development stages for drug: Pb-119
2026-07-31 10:35:34,230

,molecule_name,Pillar,Summary
0,Exenatide,IP,Exenatide is a marketed therapeutic administer...
1,Semaglutide,IP,Semaglutide is a marketed therapeutic utilizin...
2,Albenatide,IP,Albenatide is a pre-registered molecule curren...
3,Hs-20004,IP,Hs-20004 is a Phase III clinical asset facing ...
4,Liraglutide,IP,Liraglutide is a marketed GLP-1 receptor agoni...
...,...,...,...
78,Liraglutide Biobetter,IP,Liraglutide Biobetter is a Phase II clinical i...
79,Albiglutide,IP,Albiglutide is a marketed drug facing a modera...
80,Benaglutide,IP,Benaglutide is a marketed daily injectable the...
81,Dulaglutide,IP,Dulaglutide is a marketed therapeutic utilizin...


In [4]:
df.to_excel('IP Pillar Summary for GLP-1 molecules.xlsx', index=False)